# 18 — Time-series overlay

All 30 ligand-drift trajectories per target, overlaid. Tells us whether escape events are early (< 5 ns → probably a bad pose) or late (drift accumulates). That distinction feeds the bound-only hardening filter in NB 28's Claim B audit.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/18_time_series_overlay_figK.png`.)_


> **Reader guide.** *Experiment A3:* time-series overlay of ligand-drift per target across all 30
> ligands.
>
> **Method:** 30 trajectories per target overlaid, colour-coded by label.
>
> **Reproducibility contract:** reads `data/raw/complex_analyses/` timeseries parquets.

In [ ]:
# --- notebook preamble ---
NB_STEM = "34_time_series_overlay"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 6. Time-series overlay — 30 ligand drifts per target

Individual `lig_drift_A(t)` trajectories, overlaid. One panel per target. GOLD = actives, NAVY = decoys.

This is the direct view behind § 5. You can see **when** escape happens (early vs late) and whether groups of trajectories share a regime. All decoys escaping at the same time points to a force-field or PBC (periodic-boundary condition) artefact. Stochastic escape looks like real weak binding.


In [ ]:

targets = sorted(df.target.unique())
n = len(targets); nc = 3; nr = (n + nc - 1) // nc
fig, axes = plt.subplots(nr, nc, figsize=(5.0*nc, 3.3*nr), sharey=True)
for ax, tgt in zip(np.array(axes).ravel(), targets):
    for _, row in df[df.target == tgt].iterrows():
        ts_p = os.path.join(str(RAW / 'complex_analyses'), tgt, row.complex_id, 'timeseries.parquet')
        if not os.path.exists(ts_p): continue
        ts = pd.read_parquet(ts_p)
        color = NAVY; alpha = 0.30
        if 'is_active' in df and row.get('is_active') == True:
            color = GOLD; alpha = 0.85
        ax.plot(ts.time_ps/1000, ts.lig_drift_A, lw=0.8, color=color, alpha=alpha)
    ax.set_title(tgt); ax.set_xlabel('time [ns]'); ax.set_ylabel('lig drift [Å]')
    ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.4)
for ax in np.array(axes).ravel()[len(targets):]:
    ax.axis('off')
fig.suptitle('Ligand pose drift over 30 ns  ·  GOLD = active, NAVY = decoy',
             y=1.01, color=NAVY, fontweight='bold', fontsize=12)
plt.tight_layout()

**What good separation looks like.** GOLD lines cluster low (< 3 Å), NAVY lines fan upward. Targets where NAVY and GOLD interleave show poor per-complex separation. For those, aggregate features (mean/std across the 30 complexes) will usually beat any single-complex value.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
